In [ ]:
from autogen_agentchat.teams import (
    SelectorGroupChat,
)  # SelectorGroupChat: 모델을 사용하여 다음 에이전트를 선택하는 그룹 채팅 클래스
from autogen_agentchat.agents import (
    AssistantAgent,
    UserProxyAgent,
)  # UserProxyAgent: 사용자를 대신하는 프록시 에이전트로, 사용자 입력을 처리
from autogen_agentchat.conditions import (
    MaxMessageTermination,
    TextMentionTermination,
)
from autogen_agentchat.ui import Console
from autogen_ext.models.openai import OpenAIChatCompletionClient
from tools import web_search_tool, save_report_to_markdown

In [ ]:
model_client = OpenAIChatCompletionClient(model="gpt-5-mini-2025-08-07")

In [ ]:
research_planer = AssistantAgent(
    model_client=model_client,
    name="research_planer",
    description="A strategic research coordinator that breaks down complex questions into research subtasks",
    system_message="""You are a research planning specialist. Your job is to create a focused research plan.

    For each research question, create a FOCUSED research plan with:

    1. **Core Topics**: 2-3 main areas to investigate
    2. **Search Queries**: Create 3-5 specific search queries covering:
    - Latest developments and news
    - Key statistics or data
    - Expert analysis or studies
    - Future outlook

    Keep the plan focused and achievable. Quality over quantity.""",
)

research_agent = AssistantAgent(
    model_client=model_client,
    name="research_agent",
    tools=[web_search_tool],
    description="A web research specialist that searches and extracts content",
    system_message="""You are a web research specialist. Your job is to conduct focused searches based on the research plan.

    RESEARCH STRATEGY:
    1. **Execute 3-5 searches** from the research plan
    2. **Extract key information** from the results:
    - Main facts and statistics
    - Recent developments
    - Expert opinions
    - Important context

    3. **Quality focus**:
    - Prioritize authoritative sources
    - Look for recent information (within 2 years)
    - Note diverse perspectives

    After completing the searches from the plan, summarize what you found. Your goal is to gather 5-10 quality sources.""",
)

research_analyst = AssistantAgent(
    model_client=model_client,
    name="research_analyst",
    description="An expert analyst that creates research reports",
    system_message="""You are a research analyst. Create a comprehensive report from the gathered research.

    CREATE A RESEARCH REPORT with:

    ## Executive Summary
    - Key findings and conclusions
    - Main insights

    ## Background & Current State
    - Current landscape
    - Recent developments
    - Key statistics and data

    ## Analysis & Insights
    - Main trends
    - Different perspectives
    - Expert opinions

    ## Future Outlook
    - Emerging trends
    - Predictions
    - Implications

    ## Sources
    - List all sources used

    Write a clear, well-structured report based on the research gathered. End with "REPORT_COMPLETE" when finished.""",
)

quality_reviewer = AssistantAgent(
    model_client=model_client,
    name="quality_reviewer",
    tools=[save_report_to_markdown],
    description="A quality assurance specialist that evaluates research completeness and accuracy",
    system_message="""You are a quality reviewer. Your job is to check if the research analyst has produced a complete research report.

    Look for:
    - A comprehensive research report from the research analyst that ends with "REPORT_COMPLETE"
    - The research question is fully answered
    - Sources are cited and reliable
    - The report includes summary, key information, analysis, and sources

    When you see a complete research report that ends with "REPORT_COMPLETE":
    1. First, use the save_report_to_md tool to save the report to report.md
    2. Then say: "The research is complete. The report has been saved to report.md. Please review the report and let me know if you approve it or need additional research."

    If the research analyst has NOT yet created a complete report, tell them to create one now.""",
)

research_enhancer = AssistantAgent(
    model_client=model_client,
    name="research_enhancer",
    description="A specialist that identifies critical gaps only",
    system_message="""You are a research enhancement specialist. Your job is to identify ONLY CRITICAL gaps.

    Review the research and ONLY suggest additional searches if there are MAJOR gaps like:
    - Completely missing recent developments (last 6 months)
    - No statistics or data at all
    - Missing a crucial perspective that was specifically asked for

    If the research covers the basics reasonably well, say: "The research is sufficient to proceed with the report."

    Only suggest 1-2 additional searches if absolutely necessary. We prioritize getting a good report done rather than perfect coverage.""",
)

user_proxy = UserProxyAgent(
    input_func=input,
    name="user_proxy",
    description="Human reviewer who can request additional research or approve final results",
)

In [ ]:
# SelectorGroupChat에서 다음 에이전트를 선택하기 위한 프롬프트 템플릿. {roles}, {history}, {participants} 등의 플레이스홀더를 사용하여 모델에게 선택 로직을 안내함.
selector_prompt = """Choose the best agent for the current task based on the conversation history:

{roles}

Current conversation:
{history}

Available agents:
- research_planner: Plan the research approach (ONLY at the start)
- research_agent: Search for and extract content from web sources (after planning)
- research_enhancer: Identify CRITICAL gaps only (use sparingly)
- research_analyst: Write the final research report
- quality_reviewer: Check if a complete report exists
- user_proxy: Ask the human for feedback

WORKFLOW:
1. If no planning done yet → select research_planner
2. If planning done but no research → select research_agent  
3. After research_agent completes initial searches → select research_enhancer ONCE
4. If enhancer says "sufficient to proceed" → select research_analyst
5. If enhancer suggests critical searches → select research_agent ONCE more then research_analyst
6. If research_analyst said "REPORT_COMPLETE" → select quality_reviewer
7. If quality_reviewer asked for user feedback → select user_proxy

IMPORTANT: After research_agent has searched 2 times maximum, proceed to research_analyst regardless.

Pick the agent that should work next based on this workflow."""

In [ ]:
text_termination = TextMentionTermination("APPROVED")
max_message_termination = MaxMessageTermination(60)
termination_conditions = text_termination | max_message_termination

team = SelectorGroupChat(
    participants=[
        research_planer,
        research_agent,
        research_enhancer,
        research_analyst,
        quality_reviewer,
        user_proxy,
    ],
    selector_prompt=selector_prompt,
    model_client=model_client,
    allow_repeated_speaker=True,  # 동일 에이전트가 연속 발언 가능
    termination_condition=termination_conditions,
)

In [ ]:
await Console(
    team.run_stream(
        task="Research about how Microsoft uses AI to enhance developer productivity"
    )
)